# 第9章: 事前学習済み言語モデル（BERT型）

本章では、BERT型の事前学習済みモデルを利用して、マスク単語の予測や文ベクトルの計算、評判分析器（ポジネガ分類器）の構築に取り組む。

## 80. トークン化

"The movie was full of incomprehensibilities."という文をトークンに分解し、トークン列を表示せよ。

In [ ]:
!pip install tiktoken

In [ ]:
import tiktoken

text_to_tokenize = "The movie was full of incomprehensibilities."

encoding = tiktoken.get_encoding("cl100k_base")
tokens = encoding.encode(text_to_tokenize)

# 各トークンIDを文字列にデコードして表示
token_strings = [encoding.decode([t]) for t in tokens]

print(f"トークン列: {tokens}")
print(f"トークン文字列: {token_strings}")
print(f"トークン数: {len(tokens)}")

## 81. マスクの予測

"The movie was full of [MASK]."の"[MASK]"を埋めるのに最も適切なトークンを求めよ。

In [ ]:
!pip install transformers torch

In [ ]:
from transformers import pipeline

# マスク予測用のパイプラインを初期化 (英語の基本モデルを使用)
mask_filler = pipeline("fill-mask", model="bert-base-uncased")

# 予測したい文
text = "The movie was full of [MASK]."

# 予測の実行
results = mask_filler(text)

# 結果の表示
print(f"入力文: {text}\n")
for result in results:
    print(f"スコア: {result['score']:.4f}, トークン: {result['token_str']}, 文: {result['sequence']}")

## 82. マスクのtop-k予測

"The movie was full of [MASK]."の"[MASK]"に埋めるのに適切なトークン上位10個と、その確率（尤度）を求めよ。

In [ ]:
from transformers import pipeline

# マスク予測用のパイプラインを初期化
mask_filler = pipeline("fill-mask", model="bert-base-uncased")

# 予測したい文
text = "The movie was full of [MASK]."

# 上位10個の予測を取得
results = mask_filler(text, top_k=10)

# 結果の表示
print(f"入力文: {text}\n")
for i, result in enumerate(results, 1):
    print(f"{i:2}: スコア {result['score']:.4f} | トークン: {result['token_str']}")

## 83. CLSトークンによる文ベクトル

以下の文の全ての組み合わせに対して、最終層の[CLS]トークンの埋め込みベクトルを用いてコサイン類似度を求めよ。

- "The movie was full of fun."
- "The movie was full of excitement."
- "The movie was full of crap."
- "The movie was full of rubbish."


### [CLS]トークンの抽出例

各文章の先頭にある `[CLS]` トークン（ID: 101）に対応する、最終層の隠れ状態（Hidden State）を取得します。

In [ ]:
import torch
from transformers import BertTokenizer, BertModel
from sklearn.metrics.pairwise import cosine_similarity

# モデルとトークナイザーの準備
model_name = 'bert-base-uncased'
tokenizer = BertTokenizer.from_pretrained(model_name)
model = BertModel.from_pretrained(model_name)

sentences = [
    "The movie was full of fun.",
    "The movie was full of excitement.",
    "The movie was full of crap.",
    "The movie was full of rubbish."
]

# [CLS] トークンのベクトルを取得する関数
def get_cls_embedding(text):
    inputs = tokenizer(text, return_tensors="pt")
    with torch.no_grad():
        outputs = model(**inputs)
    # outputs.last_hidden_state の形状は [バッチサイズ, 系列長, 隠れ層の次元数]
    # [CLS]は先頭（インデックス0）に存在します
    cls_vec = outputs.last_hidden_state[0, 0, :]
    return cls_vec

# 各文のベクトルを計算
embeddings = [get_cls_embedding(s) for s in sentences]

# 類似度の計算と表示
print("--- コサイン類似度 ([CLS]トークン) ---")
for i in range(len(sentences)):
    for j in range(i + 1, len(sentences)):
        sim = cosine_similarity(embeddings[i].reshape(1, -1), embeddings[j].reshape(1, -1))[0][0]
        print(f"{sentences[i]} vs {sentences[j]}: {sim:.4f}")

## 84. 平均による文ベクトル

以下の文の全ての組み合わせに対して、最終層の埋め込みベクトルの平均を用いてコサイン類似度を求めよ。

- "The movie was full of fun."
- "The movie was full of excitement."
- "The movie was full of crap."
- "The movie was full of rubbish."

In [ ]:
import torch
from transformers import BertTokenizer, BertModel
from sklearn.metrics.pairwise import cosine_similarity

# モデルとトークナイザーの準備
model_name = 'bert-base-uncased'
tokenizer = BertTokenizer.from_pretrained(model_name)
model = BertModel.from_pretrained(model_name)

sentences = [
    "The movie was full of fun.",
    "The movie was full of excitement.",
    "The movie was full of crap.",
    "The movie was full of rubbish."
]

# 全てのトークンの埋め込みベクトルの平均を取得する関数
def get_mean_embedding(text):
    inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True)
    with torch.no_grad():
        outputs = model(**inputs)
    # 最終層の隠れ状態の平均を計算
    # outputs.last_hidden_state: [バッチサイズ, 系列長, 隠れ層の次元数]
    mean_vec = outputs.last_hidden_state[0].mean(dim=0)
    return mean_vec

# 各文のベクトルを計算
embeddings_mean = [get_mean_embedding(s) for s in sentences]

# 類似度の計算と表示
print("--- コサイン類似度 (全トークンの平均) ---")
for i in range(len(sentences)):
    for j in range(i + 1, len(sentences)):
        sim = cosine_similarity(embeddings_mean[i].reshape(1, -1), embeddings_mean[j].reshape(1, -1))[0][0]
        print(f"{sentences[i]} vs {sentences[j]}: {sim:.4f}")

## 85. データセットの準備

[General Language Understanding Evaluation (GLUE)](https://gluebenchmark.com/) ベンチマークで配布されている[Stanford Sentiment Treebank (SST)](https://dl.fbaipublicfiles.com/glue/data/SST-2.zip)データセットをダウンロードして、訓練データおよび開発用データのフォーマットを確認した。SST-2は、映画の評判分析のためのスタンダードな評価セットである。

In [ ]:
!pip install datasets

In [ ]:
from datasets import load_dataset

# SST-2データセットをロード
dataset = load_dataset('glue', 'sst2')

# 訓練データと開発用データの確認
print("訓練データセットの情報:")
print(f"サンプル数: {len(dataset['train'])}")
print(f"カラム: {dataset['train'].column_names}")
print(f"\n最初の3サンプル:")
for i in range(3):
    sample = dataset['train'][i]
    print(f"テキスト: {sample['sentence']}")
    print(f"ラベル: {sample['label']} (0=negative, 1=positive)")
    print()